# Sensitivity of filter feeders to spawning (`p_xspawning`, `p_lxspawning`)

Adult suspension feeders (`Y3`) spawn larvae, which become juveniles (`Yy3`).
In `src/BFM/Ben/FilterFeeder.F90` the fraction of adult biomass spawned in one event is

```
sx = p_xspawning - Ys3c / Y3c        (Ys3c: larvae already spawned)
```

and an event only happens when `sx > p_lxspawning` (and the feeders are active, food
saturation is above `p_fsat_y`, the temperature is above `p_clxTemp` and the bed shear
stress is low). The spawned biomass goes from `Y3` to `Yy3` (flux `jspaY3c`).

The OysterGrounds case uses `p_xspawning = 0.3` and `p_lxspawning = 0.2` in
`FilterFeeder.nml` (`/FilterFeeder_parameters/`). Both are single values (not per group)
and are read from the namelist, so **no recompilation is needed**.

**Lower both together.** With only `p_xspawning` reduced to 0.2 or less, `sx` can never
exceed `p_lxspawning = 0.2`, and spawning stops completely. The default scenarios keep
`p_lxspawning = 2/3 * p_xspawning`: 0.30/0.20 (case default), 0.20/0.133, 0.10/0.067.

Outputs, over one year:

* **(a)** annual mean of `param_var` (default `Y3c`) per scenario, with the % change from 0.30/0.20
* **(b)** daily values through the year, one line per scenario
* a table (printed and saved as CSV) with the annual mean of `Y3c`, `Yy3c` (juveniles)
  and `jspaY3c` (spawning flux) for every scenario

One year takes about 6 minutes per run; the runs are started in parallel.

## NaaVRE workflow

```
[spawning scenarios] --> [run GOTM spawning] --> [plot spawning]
   param_scenarios: List    run_dirs: List          results_folder: String
```

1. **spawning scenarios**: the list of scenarios, items `"xspawning/lxspawning"`.
2. **run GOTM spawning**: one model run per scenario, in parallel; outputs `run_dirs`,
   the run folders under `/tmp/data/` (the only folder shared between components).
3. **plot spawning**: reads `OG.nc` from each folder, makes the table and figure and saves
   them, with each run's `log.out` and `run_info.json`, to Cloud Storage, because NaaVRE
   clears `/tmp/data/` when the workflow finishes. It outputs that folder as `results_folder`.

In [ ]:
# spawning scenarios
# ---
# NaaVRE:
#  cell:
#   inputs: []
#   outputs:
#    - param_scenarios: List
# ...
# One item per model run: "p_xspawning/p_lxspawning".
# Keep p_lxspawning below p_xspawning, otherwise there is no spawning at all.
# The case default is 0.30/0.20; it is used as the reference for the % change.
param_scenarios = ["0.30/0.20", "0.20/0.133", "0.10/0.067"]
print("param_scenarios:", param_scenarios)

In [ ]:
# run GOTM spawning
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - param_scenarios: List
#   outputs:
#    - run_dirs: List
#   params:
#    - param_year:
#       type: Integer
#       default_value: 1972
#    - param_var:
#       type: String
#       default_value: "Y3c"
#    - param_start:
#       type: String
#       default_value: "1972-01-01 00:00:00"
#    - param_stop:
#       type: String
#       default_value: "1973-01-01 00:00:00"
# ...


# Settings
param_start = "1972-01-01 00:00:00"
param_stop = "1973-01-01 00:00:00"
param_results_root = "/tmp/data/Spawning"   # shared between workflow components
param_year = 1972            # year to average over
param_var = "Y3c"            # BFM variable in OG.nc to plot, e.g. "Yy3c" or "jspaY3c"
param_cloud_folder = "/home/jovyan/Cloud Storage/naa-vre-user-data/Spawning"   # kept after the workflow

# Spawning scenarios: labels "xspa_<x>_lxspa_<y>", one per model run.
# param_scenarios (List, from the spawning scenarios cell): items "xspawning/lxspawning",
# e.g. "0.20/0.133", or pairs [xspawning, lxspawning]; a single number x sets
# p_lxspawning to 2/3 * x.
items = param_scenarios if isinstance(param_scenarios, (list, tuple)) else str(param_scenarios).split(",")
scenarios = []
for item in items:
    if isinstance(item, (list, tuple)):
        values = [float(v) for v in item]
    elif str(item).strip():
        values = [float(v) for v in str(item).split("/")]
    else:
        continue
    xspa = values[0]
    lxspa = values[1] if len(values) > 1 else xspa * 2.0 / 3.0
    if lxspa >= xspa and xspa > 0:
        print("WARNING: p_lxspawning >= p_xspawning in {}: no spawning in this run".format(item))
    scenarios.append("xspa_{:.3f}_lxspa_{:.3f}".format(xspa, lxspa))
print("scenarios:", scenarios)

# run GOTM spawning
# Output run_dirs (List of str): run folders under param_results_root.
import os
import re
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor


def run_one(label):
    """One GOTM-BFM run for a label "xspa_<x>_lxspa_<y>"; returns the run folder."""
    parts = label.split("_")
    xspa, lxspa = float(parts[1]), float(parts[3])
    run_dir = os.path.join(param_results_root, label)
    if os.path.exists(run_dir):          # gotm-run needs a new or empty directory
        shutil.rmtree(run_dir)
    os.makedirs(param_results_root, exist_ok=True)
    command = [
        "gotm-run",
        "--out", run_dir,
        "--start", param_start,
        "--stop", param_stop,
        "--set", "FilterFeeder.nml:p_xspawning={:g}".format(xspa),
        "--set", "FilterFeeder.nml:p_lxspawning={:g}".format(lxspa),
    ]
    proc = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    status = "finished" if proc.returncode == 0 else "FAILED (see {}/log.out)".format(run_dir)
    # namelist echo in log.out: the values the model actually used
    used = ""
    log_file = os.path.join(run_dir, "log.out")
    if os.path.exists(log_file):
        with open(log_file, errors="replace") as f:
            used = " | ".join(" ".join(u.split()) for u in
                              re.findall(r"P_L?XSPAWNING\s*=[^A-Z]*", f.read(), flags=re.IGNORECASE))
    print("{:<26} {}   {}".format(label, status, used or "(values not found in log.out)"), flush=True)
    if proc.returncode != 0:
        print(proc.stdout.decode(errors="replace")[-2000:])
    return run_dir


print("Starting {} run(s) from {} to {} ...".format(len(scenarios), param_start, param_stop), flush=True)
with ThreadPoolExecutor(max_workers=max(1, len(scenarios))) as pool:
    run_dirs = list(pool.map(run_one, scenarios))
print("run_dirs:", run_dirs)

In [ ]:
# plot spawning
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - run_dirs: List
#   outputs:
#    - results_folder: String
# ...

# plot spawning
# Input run_dirs (List of str): run folders from the run cell.
# Output results_folder (str): Cloud Storage folder with the figure and the table.
import datetime
import json
import os
import shutil

import matplotlib.pyplot as plt
import netCDF4
import numpy as np

# Flatten, in case run_dirs arrives as a list of lists
folders = []
for item in (run_dirs if isinstance(run_dirs, (list, tuple)) else [run_dirs]):
    folders.extend(item if isinstance(item, (list, tuple)) else [item])

year = int(param_year)
var = str(param_var)
table_vars = [var] + [v for v in ("Y3c", "Yy3c", "jspaY3c") if v != var]

results = []    # (xspa, lxspa, dates, values of var, {table var: annual mean})
units, long_name = "", var
for run_dir in folders:
    label = os.path.basename(os.path.normpath(run_dir))
    info_file = os.path.join(run_dir, "run_info.json")
    nc_file = os.path.join(run_dir, "OG.nc")
    if os.path.exists(info_file):
        with open(info_file) as f:
            if json.load(f).get("status") != "finished":
                print("skipped (run failed):", run_dir)
                continue
    if not os.path.exists(nc_file):
        print("skipped (no OG.nc):", run_dir)
        continue
    parts = label.split("_")              # xspa_<x>_lxspa_<y>
    xspa, lxspa = float(parts[1]), float(parts[3])
    with netCDF4.Dataset(nc_file) as nc:
        t = nc.variables["time"]
        # num2date may return cftime dates, which matplotlib cannot plot:
        # convert them to standard Python datetimes
        dates = [datetime.datetime(d.year, d.month, d.day, d.hour, d.minute, d.second)
                 for d in netCDF4.num2date(t[:], t.units)]
        keep = np.array([d.year == year for d in dates])
        series = {}
        for v in table_vars:
            if v in nc.variables:
                series[v] = np.ma.filled(nc.variables[v][:].reshape(len(dates), -1)[:, 0], np.nan)[keep]
        units = getattr(nc.variables[var], "units", "")
        long_name = getattr(nc.variables[var], "long_name", var)
    annual = {v: np.nanmean(s) for v, s in series.items()}
    results.append((xspa, lxspa, [d for d, k in zip(dates, keep) if k], series[var], annual))
if not results:
    raise RuntimeError("no finished runs in {}".format(folders))
results.sort(key=lambda r: (-r[0], -r[1]))     # from high to low spawning

# Annual mean and % change from the case default (0.30/0.20, or the first run)
labels = ["x {:.2f}, lx {:.3f}".format(r[0], r[1]) for r in results]
i_ref = next((i for i, r in enumerate(results)
              if abs(r[0] - 0.3) < 1e-6 and abs(r[1] - 0.2) < 1e-6), 0)
means = np.array([r[4][var] for r in results])
change = 100.0 * (means - means[i_ref]) / means[i_ref]

print("reference:", labels[i_ref])
header = ["p_xspawning", "p_lxspawning"]
for v in table_vars:
    header += ["mean_" + v, "change_" + v + "_percent"]
table = [",".join(header)]
widths = [max(len(h), 10) for h in header]
print("  ".join(h.rjust(w) for h, w in zip(header, widths)))
for r in results:
    row = [r[0], r[1]]
    for v in table_vars:
        m = r[4].get(v, np.nan)
        m_ref = results[i_ref][4].get(v, np.nan)
        row += [m, 100.0 * (m - m_ref) / m_ref if m_ref else np.nan]
    print("  ".join("{:.4g}".format(x).rjust(w) for x, w in zip(row, widths)))
    table.append(",".join("{:.6g}".format(x) for x in row))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(results)))
x = np.arange(len(results))

ax1.bar(x, means, color=colors, width=0.6)
for xi, m, c in zip(x, means, change):
    ax1.annotate("{:+.1f}%".format(c), (xi, m), textcoords="offset points",
                 xytext=(0, 4), ha="center", fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels([l.replace(", ", "\n") for l in labels], fontsize=9)
ax1.set_xlabel("spawning fraction (p_xspawning, p_lxspawning)")
ax1.set_ylabel("annual mean {} [{}]".format(var, units))
ax1.set_title("(a) {}, mean over {}".format(long_name, year), fontsize=10)
ax1.margins(y=0.12)

for (xspa, lxspa, dates, values, annual), col, l in zip(results, colors, labels):
    ax2.plot(dates, values, color=col, label=l)
ax2.set_ylabel("{} [{}]".format(var, units))
ax2.set_title("(b) daily values in {}".format(year), fontsize=10)
ax2.legend(title="scenario", fontsize=8, frameon=False)
fig.autofmt_xdate()
plt.setp(ax1.get_xticklabels(), rotation=0, ha="center")   # keep scenario labels upright
fig.tight_layout()

# Save to Cloud Storage: /tmp/data is cleared when the workflow finishes
results_folder = str(param_cloud_folder)
os.makedirs(results_folder, exist_ok=True)
figure_file = os.path.join(results_folder, "Spawning_{}_{}.png".format(var, year))
fig.savefig(figure_file, dpi=150)
with open(os.path.join(results_folder, "Spawning_{}.csv".format(year)), "w") as f:
    f.write("\n".join(table) + "\n")
for run_dir in folders:                  # keep the model logs and run info as well
    name = os.path.basename(os.path.normpath(run_dir))
    for fname in ("log.out", "run_info.json"):
        if os.path.exists(os.path.join(run_dir, fname)):
            os.makedirs(os.path.join(results_folder, name), exist_ok=True)
            shutil.copy(os.path.join(run_dir, fname), os.path.join(results_folder, name, fname))
print("results_folder:", results_folder)
print("figure:", figure_file)
plt.show()